In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

chroma_client = Chroma(
    persist_directory="/opt/chromadb/data",
    embedding_function=embedding_model
)

In [ ]:
def retrieve_context(query, chroma_client, top_k=5):
    results = chroma_client.similarity_search(query, k=top_k)
    context = "\n".join([r.page_content for r in results])
    return context

In [ ]:
def build_prompt(query, context):
    prompt = f"""You are Amber Support Assistant, an expert in Amber molecular dynamics software.

Context:
{context}

Question:
{query}

Answer the question clearly and concisely, using a step-by-step explanation when helpful."""
    return prompt

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import torch

model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.to("cpu")

def generate_answer(prompt, max_new_tokens=300):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
def rag_pipeline(user_query):
    context = retrieve_context(user_query, chroma_client)
    prompt = build_prompt(user_query, context)
    answer = generate_answer(prompt)
    return answer

In [ ]:
if __name__ == "__main__":
    # temporary query to test if it will give a response
    query = "How does Amber handle time-series anomaly detection?"
    response = rag_pipeline(query)
    print("\nAMBER Support:\n", response)